In [3]:
# Complete Setup and Configuration
import json
import re
import logging
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from tqdm.notebook import tqdm
from collections import Counter
import pandas as pd

# Configuration Settings
MODEL_ID = "meta-llama/Meta-Llama-3-8B"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_QUANTIZATION = True

# File Paths
NER_OUTPUT_PATH = "extracted_entities_structured.json"
RELATIONS_OUTPUT_PATH = "extracted_relations_complete.json"
LABEL_STUDIO_OUTPUT = "label_studio_complete.json"
OPTIMIZED_RELATIONS_OUTPUT = "optimized_relations.json"
OPTIMIZED_LABEL_STUDIO_OUTPUT = "optimized_label_studio.json"

# Logging setup
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

print("🔧 Configuration loaded successfully")
print(f"📊 Device: {DEVICE}")
print(f"📁 Input file: {NER_OUTPUT_PATH}")
print(f"💾 Output files will be created in current directory")

🔧 Configuration loaded successfully
📊 Device: cuda
📁 Input file: extracted_entities_structured.json
💾 Output files will be created in current directory


In [5]:
# Import necessary libraries
import json
import os
import re
import pandas as pd
import numpy as np
import torch
from tqdm import tqdm
import logging
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns

# Set up logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler("few_shot_extraction.log"),
        logging.StreamHandler()
    ]
)


print("✅ Setup complete")

✅ Setup complete


In [8]:
# Import necessary libraries for LLaMA model
import json
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from tqdm import tqdm

# Define model configuration
MODEL_ID = "meta-llama/Meta-Llama-3-8B"  # You can use 'meta-llama/Llama-3-70B-Instruct' for better results
USE_QUANTIZATION = True
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NER_OUTPUT_PATH = "extracted_entities_structured.json"  # Updated to your actual file

# Initialize global variables
model = None
tokenizer = None

def load_ner_data(file_path):
    """Load and validate NER data from JSON file."""
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        
        print(f"✅ Loaded {len(data)} sentences from {file_path}")
        
        # Analyze entity types
        entity_types = {}
        for sentence in data[:10]:  # Sample first 10
            for entity in sentence['entities']:
                label = entity['label']
                entity_types[label] = entity_types.get(label, 0) + 1
        
        print(f"📊 Entity types found: {list(entity_types.keys())}")
        print(f"📋 Distribution: {entity_types}")
        return data
        
    except Exception as e:
        print(f"❌ Error loading {file_path}: {e}")
        return []

def load_llama_model():
    """Load LLaMA 3 model and tokenizer."""
    global model, tokenizer
    
    if model is not None and tokenizer is not None:
        print("✅ Using existing LLaMA model")
        return model, tokenizer
    
    print("🔄 Loading LLaMA 3 model...")
    
    try:
        # Load tokenizer
        tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        
        # Configure quantization
        if USE_QUANTIZATION:
            quantization_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_compute_dtype=torch.float16,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_use_double_quant=True
            )
            print("🔧 Using 4-bit quantization")
        else:
            quantization_config = None
        
        # Load model
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_ID,
            device_map="auto",
            quantization_config=quantization_config,
            torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
        )
        
        print(f"✅ LLaMA 3 model loaded successfully on {DEVICE}")
        return model, tokenizer
    
    except Exception as e:
        print(f"❌ Error loading LLaMA 3 model: {e}")
        raise  # Re-raise the exception to see the detailed error

# Execute loading
sentences_with_entities = load_ner_data(NER_OUTPUT_PATH)
model, tokenizer = load_llama_model()

# Display sample data
if sentences_with_entities:
    print(f"\n📝 Sample sentence:")
    sample = sentences_with_entities[0]
    print(f"Text: {sample['original_sentence']}")
    print(f"Entities ({len(sample['entities'])}):")
    for entity in sample['entities']:
        print(f"  - {entity['label']}: '{entity['text']}' [pos {entity['start_char']}:{entity['end_char']}]")

✅ Loaded 1986 sentences from extracted_entities_structured.json
📊 Entity types found: ['CHANGE', 'LOC', 'LULC', 'DATE', 'PERCENT', 'CARDINAL', 'COORDINATES']
📋 Distribution: {'CHANGE': 17, 'LOC': 7, 'LULC': 13, 'DATE': 7, 'PERCENT': 7, 'CARDINAL': 1, 'COORDINATES': 1}
🔄 Loading LLaMA 3 model...
🔧 Using 4-bit quantization


2025-06-02 11:49:15,644 - INFO - We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

✅ LLaMA 3 model loaded successfully on cuda

📝 Sample sentence:
Text: Simulation results reveal that the landscape of Thimphu city has changed considerably during the study period and the change trend is predicted to continue into 2050.
Entities (6):
  - CHANGE: 'results' [pos 11:18]
  - LOC: 'Thimphu' [pos 48:55]
  - LULC: 'city' [pos 56:60]
  - CHANGE: 'changed' [pos 65:72]
  - CHANGE: 'change' [pos 118:124]
  - DATE: '2050' [pos 161:165]


In [9]:
# High-quality few-shot examples
FEW_SHOT_EXAMPLES = [
    {
        "sentence": "Urban expansion caused forest cover to decline by 15% between 2010 and 2020.",
        "entities": [
            {"text": "Urban expansion", "label": "PROCESS", "start_char": 0, "end_char": 15},
            {"text": "forest cover", "label": "LULC", "start_char": 23, "end_char": 35},
            {"text": "decline", "label": "CHANGE", "start_char": 39, "end_char": 46},
            {"text": "15%", "label": "PERCENT", "start_char": 50, "end_char": 53},
            {"text": "2010", "label": "DATE", "start_char": 63, "end_char": 67},
            {"text": "2020", "label": "DATE", "start_char": 78, "end_char": 82}
        ],
        "relations": [
            "Urban expansion --causes--> decline",
            "decline --has_magnitude--> 15%",
            "decline --occurs_during--> 2010-2020"
        ]
    },
    {
        "sentence": "Agricultural land was converted to residential areas in Beijing during 2015-2018.",
        "entities": [
            {"text": "Agricultural land", "label": "LULC", "start_char": 0, "end_char": 17},
            {"text": "converted", "label": "CHANGE", "start_char": 22, "end_char": 31},
            {"text": "residential areas", "label": "LULC", "start_char": 35, "end_char": 52},
            {"text": "Beijing", "label": "LOC", "start_char": 56, "end_char": 63},
            {"text": "2015-2018", "label": "DATE", "start_char": 71, "end_char": 80}
        ],
        "relations": [
            "Agricultural land --converts_to--> residential areas",
            "converted --located_in--> Beijing",
            "converted --occurs_during--> 2015-2018"
        ]
    },
    {
        "sentence": "Wetlands decreased substantially in the coastal region over the past decade.",
        "entities": [
            {"text": "Wetlands", "label": "LULC", "start_char": 0, "end_char": 8},
            {"text": "decreased", "label": "CHANGE", "start_char": 9, "end_char": 18},
            {"text": "coastal region", "label": "LOC", "start_char": 40, "end_char": 54},
            {"text": "past decade", "label": "DATE", "start_char": 64, "end_char": 75}
        ],
        "relations": [
            "Wetlands --undergoes--> decreased",
            "decreased --located_in--> coastal region",
            "decreased --occurs_during--> past decade"
        ]
    },
    {
        "sentence": "The built-up area of the city expanded from 35.2% in 1990 to 52.8% in 2020 due to rapid urbanization.",
        "entities": [
            {"text": "built-up area", "label": "LULC", "start_char": 4, "end_char": 16},
            {"text": "city", "label": "LULC", "start_char": 24, "end_char": 28},
            {"text": "expanded", "label": "CHANGE", "start_char": 29, "end_char": 37},
            {"text": "35.2%", "label": "PERCENT", "start_char": 43, "end_char": 48},
            {"text": "1990", "label": "DATE", "start_char": 52, "end_char": 56},
            {"text": "52.8%", "label": "PERCENT", "start_char": 60, "end_char": 65},
            {"text": "2020", "label": "DATE", "start_char": 69, "end_char": 73},
            {"text": "urbanization", "label": "PROCESS", "start_char": 89, "end_char": 101}
        ],
        "relations": [
            "built-up area --undergoes--> expanded",
            "expanded --has_magnitude--> 35.2%",
            "expanded --has_magnitude--> 52.8%",
            "expanded --occurs_during--> 1990",
            "expanded --occurs_during--> 2020",
            "urbanization --causes--> expanded"
        ]
    },
]

print(f"✅ Loaded {len(FEW_SHOT_EXAMPLES)} few-shot examples")

✅ Loaded 4 few-shot examples


In [15]:
def construct_few_shot_prompt(sentence, entities, num_examples=2):
    """
    Construct a precise few-shot prompt for LLaMA-3 relationship extraction.
    
    Args:
        sentence: Target sentence to extract relationships from.
        entities: List of entities (each entity has 'text', 'label').
        num_examples: Number of example sentences to include in the prompt.
    
    Returns:
        str: Fully formatted few-shot prompt ready for LLaMA-3.
    """
    # Select relevant examples based on entity overlap
    selected_examples = select_relevant_examples(entities, FEW_SHOT_EXAMPLES, num_examples)
    
    # Define the prompt in LLaMA-3 instruction format
    prompt = """<|im_start|>system
You are a precise relationship extraction assistant specialized in Land Use and Land Cover (LULC) analysis.

Always follow these instructions strictly:

- Only use the format: Entity1 --relationship_type--> Entity2
- Only output relationships explicitly supported by the given sentence and entities provided.
- Only use the listed VALID_RELATIONSHIP_TYPES below.
- Do NOT add explanations, commentary, or create new entities or relationships not explicitly mentioned.

VALID_RELATIONSHIP_TYPES:
- undergoes
- converts_to
- has_magnitude
- occurs_during
- located_in
- causes
- affects
<|im_end|>

"""

    # Add selected few-shot examples
    for idx, example in enumerate(selected_examples):
        prompt += f"<|im_start|>user\n"
        prompt += f"Sentence:\n\"{example['sentence']}\"\n"
        prompt += "Entities:\n"
        for entity in example['entities']:
            prompt += f"- {entity['label']}: {entity['text']}\n"
        prompt += "Relationships:\n<|im_end|>\n"

        prompt += f"<|im_start|>assistant\n"
        for relation in example['relations']:
            prompt += f"{relation}\n"
        prompt += "<|im_end|>\n\n"

    # Add target extraction request at the end
    prompt += "<|im_start|>user\n"
    prompt += f"Sentence:\n\"{sentence}\"\n"
    prompt += "Entities:\n"
    for entity in entities:
        prompt += f"- {entity['label']}: {entity['text']}\n"
    prompt += "Relationships:\n<|im_end|>\n"
    prompt += "<|im_start|>assistant\n"

    return prompt


def select_relevant_examples(target_entities, examples, num_examples):
    """
    Select most relevant few-shot examples based on entity-label overlap.
    
    Args:
        target_entities: Entities from target sentence.
        examples: List of example dictionaries.
        num_examples: How many examples to select.
    
    Returns:
        list: Selected top-N most relevant example dictionaries.
    """
    target_labels = set(e['label'] for e in target_entities)
    
    # Score examples based on overlap in entity-label types
    scored = []
    for ex in examples:
        ex_labels = set(e['label'] for e in ex['entities'])
        overlap = len(target_labels & ex_labels)
        score = overlap / len(target_labels)
        scored.append((ex, score))
    
    # Select top examples based on scores
    scored.sort(key=lambda x: x[1], reverse=True)
    selected = [ex for ex, _ in scored[:num_examples]]
    return selected


# 🎯 Example test code execution:
if 'sentences_with_entities' in globals() and sentences_with_entities:
    test_sentence = sentences_with_entities[0]
    constructed_prompt = construct_few_shot_prompt(
        sentence=test_sentence['original_sentence'],
        entities=test_sentence['entities'],
        num_examples=2
    )
    print("\n📝 --- Generated Few-Shot Prompt (Preview) ---\n")
    print(constructed_prompt[:1500] + "\n...\n[Prompt truncated for display purposes]")
else:
    print("❗ 'sentences_with_entities' not found. Ensure you load your data first.")


📝 --- Generated Few-Shot Prompt (Preview) ---

<|im_start|>system
You are a precise relationship extraction assistant specialized in Land Use and Land Cover (LULC) analysis.

Always follow these instructions strictly:

- Only use the format: Entity1 --relationship_type--> Entity2
- Only output relationships explicitly supported by the given sentence and entities provided.
- Only use the listed VALID_RELATIONSHIP_TYPES below.
- Do NOT add explanations, commentary, or create new entities or relationships not explicitly mentioned.

VALID_RELATIONSHIP_TYPES:
- undergoes
- converts_to
- has_magnitude
- occurs_during
- located_in
- causes
- affects
<|im_end|>

<|im_start|>user
Sentence:
"Agricultural land was converted to residential areas in Beijing during 2015-2018."
Entities:
- LULC: Agricultural land
- CHANGE: converted
- LULC: residential areas
- LOC: Beijing
- DATE: 2015-2018
Relationships:
<|im_end|>
<|im_start|>assistant
Agricultural land --converts_to--> residential areas
converted

In [16]:
def extract_relations_with_llama(sentence, entities, num_examples=2):
    """
    Extract relationships using few-shot prompting with LLaMA 3 model.
    
    Args:
        sentence: Target sentence
        entities: List of entities with labels
        num_examples: Number of examples to include
        
    Returns:
        str: Raw output from model
    """
    # Construct the few-shot prompt
    prompt = construct_few_shot_prompt(sentence, entities, num_examples)
    
    # Format as LLaMA 3 chat (Instruct) format
    llama_prompt = f"<|im_start|>system\nYou are a relationship extraction expert for Land Use Land Cover analysis.\n<|im_end|>\n<|im_start|>user\n{prompt}\n<|im_end|>\n<|im_start|>assistant\n"
    
    # Generate using LLaMA 3
    try:
        # Tokenize the prompt
        inputs = tokenizer(llama_prompt, return_tensors="pt").to(DEVICE)
        
        # Check for token length and truncate if needed
        max_length = 2048
        if inputs.input_ids.size(1) > max_length:
            print(f"⚠️ Prompt too long ({inputs.input_ids.size(1)} tokens). Truncating to {max_length}.")
            inputs.input_ids = inputs.input_ids[:, :max_length]
            if 'attention_mask' in inputs:
                inputs.attention_mask = inputs.attention_mask[:, :max_length]
        
        # Generate response
        with torch.no_grad():
            outputs = model.generate(
                inputs.input_ids,
                max_new_tokens=256,
                temperature=0.1,  
                top_p=0.95,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id,
                repetition_penalty=1.2,
                attention_mask=inputs.attention_mask if 'attention_mask' in inputs else None
            )
        
        # Decode the response
        full_response = tokenizer.decode(outputs[0], skip_special_tokens=True)
        
        # Extract just the assistant's response
        assistant_response = full_response.split("<|im_start|>assistant\n")[-1].split("<|im_end|>")[0].strip()
        
        # Process the response to extract relations
        return assistant_response
        
    except Exception as e:
        print(f"❌ Error in LLaMA 3 generation: {e}")
        return f"ERROR: {str(e)}"

def parse_llama_relations(output_text, entities):
    """
    Parse the output text from LLaMA 3 into structured relations.
    
    Args:
        output_text: Model output text
        entities: List of entities with labels
        
    Returns:
        list: Structured relation objects
    """
    relations = []
    
    # Return empty list if output is empty or error
    if not output_text or "ERROR:" in output_text:
        return relations
    
    # Extract lines that match the relation pattern
    pattern = r"(.+?)\s+--(.+?)-->\s+(.+)"
    relation_lines = []
    
    for line in output_text.split('\n'):
        line = line.strip()
        # Skip if not matching relation format
        if "--" not in line or "-->" not in line:
            continue
            
        match = re.match(pattern, line)
        if match:
            source = match.group(1).strip()
            relation_type = match.group(2).strip()
            target = match.group(3).strip()
            
            relation_lines.append({
                "source": source,
                "relation_type": relation_type,
                "target": target,
                "original_text": line.strip()
            })
    
    # Match entity names to create structured relations
    for rel in relation_lines:
        # Find source entity
        source_idx = find_entity_index(rel["source"], entities)
        target_idx = find_entity_index(rel["target"], entities)
        
        if source_idx is not None and target_idx is not None:
            relations.append({
                "from_entity_idx": source_idx,
                "from_entity": entities[source_idx],
                "to_entity_idx": target_idx,
                "to_entity": entities[target_idx],
                "relation_type": rel["relation_type"],
                "confidence": "llama3_few_shot",
                "extraction_method": "llama3_few_shot"
            })
    
    return relations

def find_entity_index(text, entities):
    """
    Find the entity index based on entity text, with several fallback strategies.
    """
    text = text.lower().strip()
    
    # 1. Direct match
    for i, entity in enumerate(entities):
        if entity["text"].lower() == text:
            return i
    
    # 2. Substring match
    for i, entity in enumerate(entities):
        entity_text = entity["text"].lower()
        if (text in entity_text or entity_text in text):
            return i
    
    # 3. Fuzzy match - check if the main words match
    text_words = set(text.split())
    for i, entity in enumerate(entities):
        entity_words = set(entity["text"].lower().split())
        if len(text_words.intersection(entity_words)) > 0:
            return i
    
    return None

# Test the extraction on a sample sentence
if sentences_with_entities:
    test_sentence = sentences_with_entities[0]
    print(f"\n--- Testing LLaMA 3 Few-Shot Extraction ---\n")
    print(f"Sentence: {test_sentence['original_sentence']}")
    
    # Extract relations using LLaMA 3
    sample_output = extract_relations_with_llama(
        test_sentence['original_sentence'],
        test_sentence['entities'],
        num_examples=2
    )
    
    print("\nLLaMA 3 extraction result:")
    print(sample_output)
    
    relations = parse_llama_relations(sample_output, test_sentence['entities'])
    print(f"\nParsed {len(relations)} relations:")
    for rel in relations:
        from_text = rel["from_entity"]["text"]
        to_text = rel["to_entity"]["text"]
        rel_type = rel["relation_type"]
        print(f"- {from_text} --{rel_type}--> {to_text}")


--- Testing LLaMA 3 Few-Shot Extraction ---

Sentence: Simulation results reveal that the landscape of Thimphu city has changed considerably during the study period and the change trend is predicted to continue into 2050.

LLaMA 3 extraction result:
results --has_magnitude--> considerable
changed --located_in--> Thimphu
change --affects--> city
change --causes--> continued
continued --occurs_during--> 2050

Parsed 2 relations:
- changed --located_in--> Thimphu
- change --affects--> city


In [12]:
def process_sentences_with_llama(sentences_with_entities, num_examples=2, batch_size=10):
    """
    Process all sentences using LLaMA 3 few-shot prompting.
    
    Args:
        sentences_with_entities: List of sentences with entities
        num_examples: Number of examples to include
        batch_size: Number of sentences to process in one batch (for logging)
        
    Returns:
        list: Processing results with extracted relations
    """
    processing_results = []
    
    print(f"🚀 PROCESSING SENTENCES USING LLAMA 3 FEW-SHOT LEARNING")
    print(f"📊 Total sentences: {len(sentences_with_entities)}")
    print(f"🔧 Model: {MODEL_ID}")
    print(f"📚 Examples per prompt: {num_examples}")
    print("=" * 60)
    
    for i, sentence_data in enumerate(tqdm(sentences_with_entities, desc="LLaMA 3 extraction")):
        sentence = sentence_data['original_sentence']
        entities = sentence_data['entities']
        article_id = sentence_data.get('article_id', 'unknown')
        
        result = {
            'sentence_id': i,
            'article_id': article_id,
            'original_sentence': sentence,
            'entities': entities,
            'llama_raw_output': '',
            'final_relations': [],
            'method_used': 'llama3_few_shot',
            'selected_examples': [],
            'error': None
        }
        
        try:
            # Skip sentences with insufficient entities
            if len(entities) < 2:
                result['error'] = 'Insufficient entities (<2)'
                result['method_used'] = 'skipped'
                processing_results.append(result)
                continue
            
            # Select examples (for logging)
            selected_examples = select_relevant_examples(entities, FEW_SHOT_EXAMPLES, num_examples)
            result['selected_examples'] = [ex["sentence"] for ex in selected_examples]
            
            # Extract relations using LLaMA 3
            llama_output = extract_relations_with_llama(
                sentence,
                entities,
                num_examples=num_examples
            )
            
            result['llama_raw_output'] = llama_output
            
            # Parse the output to structured relations
            relations = parse_llama_relations(llama_output, entities)
            result['final_relations'] = relations
            
            # Log detailed progress for batch_size samples
            if i % batch_size == 0:
                print(f"\n📝 Sample - Sentence {i}:")
                print(f"Text: {sentence[:100]}...")
                print(f"Entities: {len(entities)}")
                print(f"Extracted Relations: {len(relations)}")
                if relations:
                    for j, rel in enumerate(relations[:3]):  # Show first 3 relations
                        from_text = rel["from_entity"]["text"]
                        to_text = rel["to_entity"]["text"]
                        rel_type = rel["relation_type"]
                        print(f"  {j+1}. {from_text} --{rel_type}--> {to_text}")
        
        except Exception as e:
            result['error'] = str(e)
            result['method_used'] = 'error'
            print(f"Error processing sentence {i}: {e}")
        
        processing_results.append(result)
    
    print(f"\n✅ Processing complete!")
    print(f"📊 Processed {len(processing_results)} sentences")
    
    # Calculate success rate
    successful = sum(1 for r in processing_results if r['final_relations'])
    print(f"📈 Success rate: {successful}/{len(processing_results)} ({successful/len(processing_results)*100:.1f}%)")
    
    return processing_results

def run_llama_pipeline(num_sentences=None, num_examples=2):
    """
    Run the complete LLaMA 3 extraction pipeline.
    
    Args:
        num_sentences: Number of sentences to process (None for all)
        num_examples: Number of examples to include in prompts
    """
    global sentences_with_entities
    
    print("🚀 RUNNING COMPLETE LLAMA 3 FEW-SHOT EXTRACTION PIPELINE")
    print("=" * 70)
    
    # Prepare data
    if num_sentences:
        data_to_process = sentences_with_entities[:num_sentences]
        print(f"📝 Processing {len(data_to_process)} sentences (sample)")
    else:
        data_to_process = sentences_with_entities
        print(f"📝 Processing all {len(data_to_process)} sentences")
    
    # Step 1: Process sentences with LLaMA 3
    results = process_sentences_with_llama(
        data_to_process,
        num_examples=num_examples
    )
    
    # Step 2: Save results and create Label Studio files
    output_path = "output/llama3_few_shot_results.json"
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(results, f, indent=2, ensure_ascii=False)
    print(f"✅ Results saved to {output_path}")
    
    # Step 3: Create Label Studio predictions file
    label_studio_tasks = save_results_and_create_label_studio_predictions(results)
    
    # Step 4: Analyze results
    analyze_few_shot_results(results)
    
    print("\n🎯 PIPELINE EXECUTION COMPLETE!")
    print(f"📁 Results saved to: {output_path}")
    
    return results, label_studio_tasks

# Note: Uncomment the next line to run the LLaMA 3 pipeline
# llama_results, label_studio = run_llama_pipeline(num_sentences=10)

In [13]:
def save_results_and_create_label_studio_predictions(processing_results):
    """
    Save detailed results and create Label Studio files in predictions format.
    """
    PREDICTIONS_OUTPUT_PATH = os.path.join(OUTPUT_PATH, "few_shot_label_studio_predictions.json")
    
    print("💾 SAVING FEW-SHOT RESULTS AND CREATING LABEL STUDIO FILES")
    print("=" * 70)
    
    # Save detailed processing results
    with open(FEW_SHOT_OUTPUT, 'w', encoding='utf-8') as f:
        json.dump(processing_results, f, indent=2, ensure_ascii=False)
    
    print(f"✅ Detailed results saved: {FEW_SHOT_OUTPUT}")
    
    # Create Label Studio tasks in predictions format
    label_studio_tasks = []
    
    for result_idx, result in enumerate(processing_results):
        sentence = result['original_sentence']
        entities = result['entities']
        relations = result['final_relations']
        
        prediction_result_items = []
        entity_id_map = {}  # Maps entity index to prediction ID
        
        # 1. Create entity predictions
        for i, entity in enumerate(entities):
            unique_entity_pred_id = f"ent_pred_{result_idx}_{i}"
            entity_id_map[i] = unique_entity_pred_id
            
            entity_prediction = {
                "value": {
                    "start": entity['start_char'],
                    "end": entity['end_char'],
                    "text": entity['text'],
                    "labels": [entity['label']]
                },
                "id": unique_entity_pred_id,
                "from_name": "label",
                "to_name": "text",
                "type": "labels",
                "score": 0.95  # Example score
            }
            prediction_result_items.append(entity_prediction)
        
        # 2. Create relation predictions
        for j, relation in enumerate(relations):
            source_entity_pred_id = entity_id_map.get(relation['from_entity_idx'])
            target_entity_pred_id = entity_id_map.get(relation['to_entity_idx'])
            
            if source_entity_pred_id and target_entity_pred_id:
                relation_prediction = {
                    "from_id": source_entity_pred_id,
                    "to_id": target_entity_pred_id,
                    "type": "relation",
                    "direction": "right",
                    "labels": [relation['relation_type']],
                    "from_name": "relation",
                    "to_name": "label",
                    "score": 0.85  # Example score
                }
                prediction_result_items.append(relation_prediction)
        
        # Create the top-level task with predictions
        formatted_task = {
            "data": {
                "text": sentence
            },
            "predictions": [{
                "model_version": "few-shot-model-v1.0",
                "score": 0.9,
                "result": prediction_result_items
            }],
            "id": result['sentence_id'],
            "meta": {
                "article_id": result['article_id'],
                "method_used": result['method_used'],
                "num_entities": len(entities),
                "num_relations": len(relations)
            }
        }
        
        label_studio_tasks.append(formatted_task)
    
    # Save Label Studio predictions file
    with open(PREDICTIONS_OUTPUT_PATH, 'w', encoding='utf-8') as f:
        json.dump(label_studio_tasks, f, indent=2, ensure_ascii=False)
    
    print(f"✅ Label Studio predictions file saved: {PREDICTIONS_OUTPUT_PATH}")
    print(f"📊 Created {len(label_studio_tasks)} Label Studio tasks")
    
    # Create a sample Label Studio configuration XML
    LABEL_STUDIO_CONFIG_PATH = os.path.join(OUTPUT_PATH, "label_studio_config.xml")
    
    label_studio_config = """<View>
  <Text name="text" value="$text"/>
  <Labels name="label" toName="text">
    <Label value="CHANGE" background="#FF6B6B"/>
    <Label value="LULC" background="#4ECDC4"/>
    <Label value="LOC" background="#45B7D1"/>
    <Label value="DATE" background="#96CEB4"/>
    <Label value="PERCENT" background="#FECA57"/>
    <Label value="PROCESS" background="#FF9FF3"/>
  </Labels>
  <Relations name="relation" toName="label">
    <Relation value="undergoes" background="#FF6B6B"/>
    <Relation value="occurs_during" background="#4ECDC4"/>
    <Relation value="located_in" background="#45B7D1"/>
    <Relation value="causes" background="#FF9FF3"/>
    <Relation value="has_magnitude" background="#FECA57"/>
    <Relation value="converts_to" background="#96CEB4"/>
    <Relation value="affects" background="#A55EEA"/>
  </Relations>
</View>"""
    
    with open(LABEL_STUDIO_CONFIG_PATH, 'w', encoding='utf-8') as f:
        f.write(label_studio_config)
    
    print(f"✅ Label Studio configuration saved: {LABEL_STUDIO_CONFIG_PATH}")
    
    return label_studio_tasks

def analyze_few_shot_results(processing_results):
    """
    Analyze the few-shot extraction results.
    """
    print("\n📊 FEW-SHOT EXTRACTION ANALYSIS")
    print("=" * 50)
    
    # Basic statistics
    total_sentences = len(processing_results)
    successful = sum(1 for r in processing_results if r['final_relations'])
    total_relations = sum(len(r['final_relations']) for r in processing_results if r['final_relations'])
    avg_relations = total_relations / successful if successful > 0 else 0
    
    # Error analysis
    errors = [r for r in processing_results if r.get('error')]
    error_types = {}
    for r in errors:
        error_msg = r['error']
        error_category = 'Insufficient entities' if 'Insufficient' in error_msg else 'Processing error'
        error_types[error_category] = error_types.get(error_category, 0) + 1
    
    # Relation type distribution
    relation_types = {}
    for r in processing_results:
        for relation in r.get('final_relations', []):
            rel_type = relation['relation_type']
            relation_types[rel_type] = relation_types.get(rel_type, 0) + 1
    
    # Print analysis
    print(f"📈 FEW-SHOT PERFORMANCE:")
    print(f"- Total sentences: {total_sentences}")
    print(f"- Successful extractions: {successful} ({successful/total_sentences*100:.1f}%)")
    print(f"- Total relations extracted: {total_relations}")
    print(f"- Average relations per successful sentence: {avg_relations:.2f}")
    
    print(f"\n🔗 RELATION TYPE DISTRIBUTION:")
    for rel_type, count in sorted(relation_types.items(), key=lambda x: x[1], reverse=True):
        print(f"- {rel_type}: {count} instances ({count/total_relations*100:.1f}%)")
    
    print(f"\n❌ ERROR ANALYSIS:")
    for error_type, count in error_types.items():
        print(f"- {error_type}: {count} ({count/len(errors)*100:.1f}% of errors)")
    
    # Save analysis
    analysis = {
        "performance": {
            "total_sentences": total_sentences,
            "successful_extractions": successful,
            "success_rate": f"{successful/total_sentences*100:.1f}%",
            "total_relations": total_relations,
            "avg_relations_per_sentence": avg_relations
        },
        "relation_types": relation_types,
        "error_analysis": error_types
    }
    
    output_file = os.path.join(OUTPUT_PATH, "few_shot_analysis.json")
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(analysis, f, indent=2, ensure_ascii=False)
    
    print(f"\n✅ Analysis saved: {output_file}")
    
    return analysis

# Just declare the functions - don't execute here
print("✅ Result saving and analysis functions defined")

✅ Result saving and analysis functions defined


In [14]:
def run_complete_pipeline(num_sentences=None, model_type="openai", num_examples=2):
    """
    Run the complete few-shot extraction pipeline.
    
    Args:
        num_sentences: Number of sentences to process (None for all)
        model_type: 'openai' or 'local'
        num_examples: Number of examples to include in prompts
    """
    global sentences_with_entities
    
    print("🚀 RUNNING COMPLETE FEW-SHOT EXTRACTION PIPELINE")
    print("=" * 70)
    
    # Prepare data
    if num_sentences:
        data_to_process = sentences_with_entities[:num_sentences]
        print(f"📝 Processing {len(data_to_process)} sentences (sample)")
    else:
        data_to_process = sentences_with_entities
        print(f"📝 Processing all {len(data_to_process)} sentences")
    
    # Step 1: Process sentences with few-shot
    results = process_sentences_with_few_shot(
        data_to_process,
        model_type=model_type,
        num_examples=num_examples
    )
    
    # Step 2: Save results and create Label Studio files
    label_studio_tasks = save_results_and_create_label_studio_predictions(results)
    
    # Step 3: Analyze results
    analysis = analyze_few_shot_results(results)
    
    print("\n🎯 PIPELINE EXECUTION COMPLETE!")
    print(f"📁 Results and analysis files saved to: {OUTPUT_PATH}")
    print(f"📊 Successfully extracted relationships for {analysis['performance']['successful_extractions']}/{len(results)} sentences")
    
    return results, label_studio_tasks, analysis

# Note: Uncomment the next line to run the complete pipeline
# all_results, label_studio, analysis = run_complete_pipeline(num_sentences=10)

In [ ]:
def save_mistral_results_and_create_label_studio_predictions(processing_results, output_dir="output"):
    """
    Save Mistral relation extraction results and create Label Studio files in predictions format.
    """
    import os
    
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    MISTRAL_OUTPUT = os.path.join(output_dir, "mistral_relation_extraction_results.json")
    PREDICTIONS_OUTPUT_PATH = os.path.join(output_dir, "mistral_label_studio_predictions.json")
    
    print("💾 SAVING MISTRAL RESULTS AND CREATING LABEL STUDIO FILES")
    print("=" * 70)
    
    # Save detailed processing results
    with open(MISTRAL_OUTPUT, 'w', encoding='utf-8') as f:
        json.dump(processing_results, f, indent=2, ensure_ascii=False)
    
    print(f"✅ Detailed results saved: {MISTRAL_OUTPUT}")
    
    # Create Label Studio tasks in predictions format
    label_studio_tasks = []
    
    for result_idx, result in enumerate(processing_results):
        sentence = result['sentence']
        entities = result['entities']
        relations = result['relations']  # From your Mistral extraction
        
        prediction_result_items = []
        entity_id_map = {}  # Maps entity text to prediction ID
        
        # 1. Create entity predictions
        for i, entity in enumerate(entities):
            unique_entity_pred_id = f"mistral_ent_{result_idx}_{i}"
            entity_id_map[entity['text']] = unique_entity_pred_id
            
            entity_prediction = {
                "value": {
                    "start": entity['start_char'],
                    "end": entity['end_char'],
                    "text": entity['text'],
                    "labels": [entity['label']]
                },
                "id": unique_entity_pred_id,
                "from_name": "label",
                "to_name": "text",
                "type": "labels",
                "score": 0.95  # High confidence for pre-identified entities
            }
            prediction_result_items.append(entity_prediction)
        
        # 2. Create relation predictions from Mistral output
        for j, relation in enumerate(relations):
            # Map relation confidence to score
            confidence_score = {
                "HIGH": 0.9,
                "MEDIUM": 0.7,
                "LOW": 0.5,
                "UNKNOWN": 0.6
            }.get(relation.get('confidence', 'UNKNOWN'), 0.6)
            
            source_entity_pred_id = entity_id_map.get(relation['source'])
            target_entity_pred_id = entity_id_map.get(relation['target'])
            
            if source_entity_pred_id and target_entity_pred_id:
                relation_prediction = {
                    "from_id": source_entity_pred_id,
                    "to_id": target_entity_pred_id,
                    "type": "relation",
                    "direction": "right",
                    "labels": [relation['relationship']],  # Dynamic relation types from Mistral
                    "from_name": "relation",
                    "to_name": "label",
                    "score": confidence_score,
                    "meta": {
                        "confidence": relation.get('confidence', 'UNKNOWN'),
                        "discovered_by": "mistral"
                    }
                }
                prediction_result_items.append(relation_prediction)
        
        # Create the top-level task with predictions
        formatted_task = {
            "data": {
                "text": sentence
            },
            "predictions": [{
                "model_version": "mistral-relation-discovery-v1.0",
                "score": sum(rel.get('score', 0.6) for rel in prediction_result_items if rel.get('type') == 'relation') / max(1, len([r for r in prediction_result_items if r.get('type') == 'relation'])),
                "result": prediction_result_items
            }],
            "id": result.get('sentence_id', result_idx),
            "meta": {
                "method_used": "mistral_open_discovery",
                "num_entities": len(entities),
                "num_relations": len(relations),
                "raw_model_response": result.get('raw_response', '')[:200] + '...' if result.get('raw_response', '') else ''
            }
        }
        
        label_studio_tasks.append(formatted_task)
    
    # Save Label Studio predictions file
    with open(PREDICTIONS_OUTPUT_PATH, 'w', encoding='utf-8') as f:
        json.dump(label_studio_tasks, f, indent=2, ensure_ascii=False)
    
    print(f"✅ Label Studio predictions file saved: {PREDICTIONS_OUTPUT_PATH}")
    print(f"📊 Created {len(label_studio_tasks)} Label Studio tasks")
    
    # Create dynamic Label Studio configuration based on discovered relations
    create_dynamic_label_studio_config(processing_results, output_dir)
    
    return label_studio_tasks

def create_dynamic_label_studio_config(processing_results, output_dir):
    """Create Label Studio config with dynamically discovered relation types."""
    
    # Extract all unique relation types discovered by Mistral
    all_relation_types = set()
    all_entity_types = set()
    
    for result in processing_results:
        for relation in result.get('relations', []):
            all_relation_types.add(relation['relationship'])
        for entity in result.get('entities', []):
            all_entity_types.add(entity['label'])
    
    # Color palette for relations
    colors = [
        "#FF6B6B", "#4ECDC4", "#45B7D1", "#96CEB4", "#FECA57", 
        "#FF9FF3", "#A55EEA", "#54A0FF", "#5F27CD", "#00D2D3",
        "#FF9F43", "#10AC84", "#EE5A24", "#0098C7", "#8395A7"
    ]
    
    LABEL_STUDIO_CONFIG_PATH = os.path.join(output_dir, "mistral_label_studio_config.xml")
    
    # Build entity labels
    entity_labels = ""
    for i, entity_type in enumerate(sorted(all_entity_types)):
        color = colors[i % len(colors)]
        entity_labels += f'    <Label value="{entity_type}" background="{color}"/>\n'
    
    # Build relation labels
    relation_labels = ""
    for i, relation_type in enumerate(sorted(all_relation_types)):
        color = colors[i % len(colors)]
        relation_labels += f'    <Relation value="{relation_type}" background="{color}"/>\n'
    
    label_studio_config = f"""<View>
  <Text name="text" value="$text"/>
  <Labels name="label" toName="text">
{entity_labels}  </Labels>
  <Relations name="relation" toName="label">
{relation_labels}  </Relations>
</View>"""
    
    with open(LABEL_STUDIO_CONFIG_PATH, 'w', encoding='utf-8') as f:
        f.write(label_studio_config)
    
    print(f"✅ Dynamic Label Studio configuration saved: {LABEL_STUDIO_CONFIG_PATH}")
    print(f"📋 Configured {len(all_entity_types)} entity types and {len(all_relation_types)} relation types")

def analyze_mistral_results(processing_results, output_dir="output"):
    """
    Analyze the Mistral relation extraction results.
    """
    print("\n📊 MISTRAL RELATION EXTRACTION ANALYSIS")
    print("=" * 50)
    
    # Basic statistics
    total_sentences = len(processing_results)
    successful = sum(1 for r in processing_results if r.get('relations') and len(r['relations']) > 0)
    total_relations = sum(len(r.get('relations', [])) for r in processing_results)
    avg_relations = total_relations / successful if successful > 0 else 0
    
    # Confidence distribution
    confidence_dist = {"HIGH": 0, "MEDIUM": 0, "LOW": 0, "UNKNOWN": 0}
    for r in processing_results:
        for relation in r.get('relations', []):
            conf = relation.get('confidence', 'UNKNOWN')
            confidence_dist[conf] = confidence_dist.get(conf, 0) + 1
    
    # Relation type distribution (discovered types)
    relation_types = {}
    for r in processing_results:
        for relation in r.get('relations', []):
            rel_type = relation['relationship']
            relation_types[rel_type] = relation_types.get(rel_type, 0) + 1
    
    # Print analysis
    print(f"📈 MISTRAL PERFORMANCE:")
    print(f"- Total sentences: {total_sentences}")
    print(f"- Successful extractions: {successful} ({successful/total_sentences*100:.1f}%)")
    print(f"- Total relations extracted: {total_relations}")
    print(f"- Average relations per successful sentence: {avg_relations:.2f}")
    
    print(f"\n🎯 CONFIDENCE DISTRIBUTION:")
    for conf_level, count in confidence_dist.items():
        percentage = (count/total_relations*100) if total_relations > 0 else 0
        print(f"- {conf_level}: {count} ({percentage:.1f}%)")
    
    print(f"\n🔗 DISCOVERED RELATION TYPES (Top 15):")
    sorted_relations = sorted(relation_types.items(), key=lambda x: x[1], reverse=True)
    for rel_type, count in sorted_relations[:15]:
        print(f"- {rel_type}: {count} instances ({count/total_relations*100:.1f}%)")
    
    # Save analysis
    analysis = {
        "performance": {
            "total_sentences": total_sentences,
            "successful_extractions": successful,
            "success_rate": f"{successful/total_sentences*100:.1f}%",
            "total_relations": total_relations,
            "avg_relations_per_sentence": avg_relations
        },
        "confidence_distribution": confidence_dist,
        "discovered_relation_types": relation_types,
        "unique_relation_types_count": len(relation_types)
    }
    
    output_file = os.path.join(output_dir, "mistral_analysis.json")
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(analysis, f, indent=2, ensure_ascii=False)
    
    print(f"\n✅ Analysis saved: {output_file}")
    print(f"📊 Discovered {len(relation_types)} unique relation types!")
    
    return analysis

# Usage after processing your data
def complete_mistral_pipeline_with_saving(sentences_data, model, tokenizer, max_sentences=None):
    """Complete pipeline with saving in Label Studio format."""
    
    # Step 1: Process sentences (your existing function)
    results = process_all_sentences(sentences_data, model, tokenizer, max_sentences)
    
    # Step 2: Save in Label Studio format
    label_studio_tasks = save_mistral_results_and_create_label_studio_predictions(results)
    
    # Step 3: Analyze results
    analysis = analyze_mistral_results(results)
    
    print(f"\n🎉 PIPELINE COMPLETE!")
    print(f"📁 Check the 'output' folder for:")
    print(f"   - mistral_relation_extraction_results.json (detailed results)")
    print(f"   - mistral_label_studio_predictions.json (Label Studio format)")
    print(f"   - mistral_label_studio_config.xml (Label Studio configuration)")
    print(f"   - mistral_analysis.json (performance analysis)")
    
    return results, label_studio_tasks, analysis

print("✅ Mistral Label Studio saving functions defined")